In [ ]:
import planetary_computer
import itertools
import dask.dataframe as dd

cc = planetary_computer.get_container_client("pcstacitems", "items")

blobs = list(cc.list_blobs("sentinel-2-l2a.parquet/"))

# def key(blob):
#     return blob.name.split("/")[1].split("_")[0]

# keep_blobs = []
# for k, v in itertools.groupby(sorted(blobs, key=key), key=key):
#     v = list(v)
#     # blob = max(v, key=lambda x: x.last_modified)
#     keep_blobs.append(blob)
    
# uris = [f"az://items/{blob.name}" for blob in keep_blobs]

In [40]:
# Date filters
max_date = "2021-01-01"
min_date = "2015-01-01"

# Grab blobs that are within the date range
keep_blobs = []
for blob in blobs:
    date = blob.name.split("/")[1].split("_")[1]
    if min_date <= date <= max_date:
        keep_blobs.append(blob)
        print(blob.name)

uris = [f"az://items/{blob.name}" for blob in keep_blobs]

sentinel-2-l2a.parquet/part-0000_2015-07-04T10:10:06.027000+00:00_2015-07-31T10:14:26.027001+00:00.parquet
sentinel-2-l2a.parquet/part-0001_2015-08-01T01:31:06.027000+00:00_2015-08-31T11:34:06.027001+00:00.parquet
sentinel-2-l2a.parquet/part-0002_2015-09-01T14:14:56.027000+00:00_2015-09-30T11:11:06.027001+00:00.parquet
sentinel-2-l2a.parquet/part-0003_2015-10-01T01:01:36.027000+00:00_2015-10-28T05:47:02.027001+00:00.parquet
sentinel-2-l2a.parquet/part-0004_2015-11-11T16:55:22.031000+00:00_2015-11-30T22:58:22.031001+00:00.parquet
sentinel-2-l2a.parquet/part-0005_2015-12-01T00:31:32.031000+00:00_2015-12-31T23:58:02.030001+00:00.parquet
sentinel-2-l2a.parquet/part-0006_2016-01-01T00:02:42.030000+00:00_2016-01-31T23:48:12.030001+00:00.parquet
sentinel-2-l2a.parquet/part-0007_2016-02-01T00:48:42.031000+00:00_2016-02-22T06:59:01.030001+00:00.parquet
sentinel-2-l2a.parquet/part-0008_2016-03-01T14:47:42.030000+00:00_2016-03-31T23:28:42.030001+00:00.parquet
sentinel-2-l2a.parquet/part-0009_2016

In [41]:
len(uris)

66

In [ ]:
df = dd.read_parquet(uris,
                     columns=['id', 'geometry', 'datetime', 'bbox', 'eo:cloud_cover', "s2:granule_id", "s2:nodata_pixel_percentage", "s2:saturated_defective_pixel_percentage"],
                     storage_options={"account_name": "pcstacitems", "credential": planetary_computer.sas.get_token("pcstacitems", "items").token})
df.head()

,id,geometry,datetime,bbox,eo:cloud_cover,s2:granule_id,s2:nodata_pixel_percentage,s2:saturated_defective_pixel_percentage
0,S2A_MSIL2A_20150704T101006_R022_T35XQA_2021041...,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00\x1e\x00...",2015-07-04 10:10:06.027000+00:00,"{'xmin': 32.782321671748434, 'ymin': 71.805561...",97.851394,S2A_OPER_MSI_L2A_TL_ESRI_20210411T133711_A0001...,20.722227,0.0
1,S2A_MSIL2A_20150704T101006_R022_T32TMM_2021041...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x0e\x00...,2015-07-04 10:10:06.027000+00:00,"{'xmin': 8.479220766509275, 'ymin': 41.4606964...",0.177088,S2A_OPER_MSI_L2A_TL_ESRI_20210411T133213_A0001...,63.654864,0.0
2,S2A_MSIL2A_20150715T112846_R037_T29TPL_2021041...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00 \x00\x0...,2015-07-15 11:28:46.027000+00:00,"{'xmin': -7.7299243, 'ymin': 45.0432611, 'xmax...",98.440793,S2A_OPER_MSI_L2A_TL_ESRI_20210411T152630_A0003...,96.404278,0.0
3,S2A_MSIL2A_20150704T101006_R022_T36WWC_2021041...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\t\x00\x...,2015-07-04 10:10:06.027000+00:00,"{'xmin': 32.99946824095416, 'ymin': 69.6891048...",88.028410,S2A_OPER_MSI_L2A_TL_ESRI_20210411T133717_A0001...,88.475966,0.0
4,S2A_MSIL2A_20150704T101006_R022_T31RGK_2021041...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,2015-07-04 10:10:06.027000+00:00,"{'xmin': 4.999786508901344, 'ymin': 26.0978038...",45.840056,S2A_OPER_MSI_L2A_TL_ESRI_20210411T132936_A0001...,0.000000,0.0


In [43]:
df.columns

Index(['id', 'geometry', 'datetime', 'bbox', 'eo:cloud_cover', 's2:granule_id',
       's2:nodata_pixel_percentage',
       's2:saturated_defective_pixel_percentage'],
      dtype='object')

In [44]:
print(len(df))

14728217


In [45]:
# df = df[['id', 'geometry', 'bbox', 'datetime', 'eo:cloud_cover', "s2:product_uri", "s2:granule_id", "s2:nodata_pixel_percentage", "s2:saturated_defective_pixel_percentage"]]
df = df[['id', 'geometry', 'datetime', 'eo:cloud_cover', "s2:granule_id", "s2:nodata_pixel_percentage", "s2:saturated_defective_pixel_percentage"]]

df["datetime"] = dd.to_datetime(df["datetime"])

In [7]:
# df.head()

In [46]:
filtered_df = df[(df['datetime'] < max_date) &
                 (df['datetime'] >= min_date) &
                 (df['eo:cloud_cover'] < 20) &
                 (df['s2:nodata_pixel_percentage'] < 10) &
                 (df['s2:saturated_defective_pixel_percentage'] < 10)]

In [47]:
filtered_df = filtered_df.compute()
print(len(filtered_df))

2332449


In [48]:
filtered_df.shape

(2332449, 7)

In [33]:
filtered_df.head()

,id,geometry,datetime,eo:cloud_cover,s2:granule_id,s2:nodata_pixel_percentage,s2:saturated_defective_pixel_percentage
14,S2A_MSIL2A_20200119T165621_R026_T15SXC_2020100...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,2020-01-19 16:56:21.024000+00:00,0.778711,S2A_OPER_MSI_L2A_TL_ESRI_20201002T204310_A0239...,0.000056,0.0
16,S2B_MSIL2A_20200104T083239_R021_T33HYE_2020100...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,2020-01-04 08:32:39.024000+00:00,6.143848,S2B_OPER_MSI_L2A_TL_ESRI_20201002T225431_A0147...,0.000000,0.0
30,S2B_MSIL2A_20200102T161649_R140_T16QFL_2020100...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,2020-01-02 16:16:49.024000+00:00,9.720402,S2B_OPER_MSI_L2A_TL_ESRI_20201002T220913_A0147...,0.000000,0.0
41,S2B_MSIL2A_20200105T112349_R037_T29TNG_2020100...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\n\x00\x...,2020-01-05 11:23:49.024000+00:00,9.708129,S2B_OPER_MSI_L2A_TL_ESRI_20201002T232257_A0147...,3.455304,0.0
45,S2A_MSIL2A_20200105T004701_R102_T54JUT_2020100...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,2020-01-05 00:47:01.024000+00:00,0.011214,S2A_OPER_MSI_L2A_TL_ESRI_20201002T230337_A0236...,0.000000,0.0


In [49]:
max_date = filtered_df['datetime'].max().strftime("%Y_%m_%d")
print(max_date)

2020_12_31


In [50]:
min_date = filtered_df['datetime'].min().strftime("%Y_%m_%d")
print(min_date)

2015_07_04


In [51]:
filtered_df.to_parquet(f"s2l2a_clouds_lt_{max_date}_gt_{min_date}.parquet", index=False)
print(f"Saved to: s2l2a_clouds_lt_{max_date}_gt_{min_date}.parquet")

Saved to: s2l2a_clouds_lt_2020_12_31_gt_2015_07_04.parquet
